In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Start Spark
spark = SparkSession.builder \
    .appName("Sales Capstone - End to End") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

sales_data = [
    ("TXN001","Delhi ","Laptop","Electronics","45000","2024-01-05","Compl"),
    ("TXN002","Mumbai","Mobile ","electronics","32000","05/01/2024","Comp"),
    ("TXN003","Bangalore","Tablet"," Electronics ","30000","2024/01/06",""),
    ("TXN004","Delhi","Laptop","Electronics","","2024-01-07","Cancelled"),
    ("TXN005","Chennai","Mobile","Electronics","invalid","2024-01-08","Co"),
    ("TXN006","Mumbai","Tablet","Electronics",None,"2024-01-08","Complete"),
    ("TXN007","Delhi","Laptop","electronics","45000","09-01-2024","Comple"),
    ("TXN008","Bangalore","Mobile","Electronics","28000","2024-01-09","Co"),
    ("TXN009","Mumbai","Laptop","Electronics","55000","2024-01-10","Compl"),
    ("TXN009","Mumbai","Laptop","Electronics","55000","2024-01-10","Compl"),
]

customer_data = [
    ("C001","Delhi","Premium"),
    ("C002","Mumbai","Standard"),
    ("C003","Bangalore","Premium"),
    ("C004","Chennai","Standard"),
    ("C005","Mumbai","Premium"),
]

city_lookup = [
    ("Delhi","Tier-1"),
    ("Mumbai","Tier-1"),
    ("Bangalore","Tier-1"),
    ("Chennai","Tier-2"),
]





PART 1
1. Create schemas

In [2]:

sales_schema = StructType([
    StructField("txn_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", StringType(), True),   # will parse to Integer
    StructField("txn_date", StringType(), True), # will parse to Date
    StructField("status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
])

city_lookup_schema = StructType([
    StructField("city", StringType(), True),
    StructField("tier", StringType(), True),
])



2.Load into data frames

In [4]:
sales_data = [
    ("TXN001","Delhi ","Laptop","Electronics","45000","2024-01-05","Compl"),
    ("TXN002","Mumbai","Mobile ","electronics","32000","05/01/2024","Comp"),
    ("TXN003","Bangalore","Tablet"," Electronics ","30000","2024/01/06",""),
    ("TXN004","Delhi","Laptop","Electronics","","2024-01-07","Cancelled"),
    ("TXN005","Chennai","Mobile","Electronics","invalid","2024-01-08","Co"),
    ("TXN006","Mumbai","Tablet","Electronics",None,"2024-01-08","Complete"),
    ("TXN007","Delhi","Laptop","electronics","45000","09-01-2024","Comple"),
    ("TXN008","Bangalore","Mobile","Electronics","28000","2024-01-09","Co"),
    ("TXN009","Mumbai","Laptop","Electronics","55000","2024-01-10","Compl"),
    ("TXN009","Mumbai","Laptop","Electronics","55000","2024-01-10","Compl"),
]

customer_data = [
    ("C001","Delhi","Premium"),
    ("C002","Mumbai","Standard"),
    ("C003","Bangalore","Premium"),
    ("C004","Chennai","Standard"),
    ("C005","Mumbai","Premium"),
]

city_lookup = [
    ("Delhi","Tier-1"),
    ("Mumbai","Tier-1"),
    ("Bangalore","Tier-1"),
    ("Chennai","Tier-2"),
]

sales_raw = spark.createDataFrame(sales_data, schema=sales_schema)
customer_raw = spark.createDataFrame(customer_data, schema=customer_schema)
city_lookup_raw = spark.createDataFrame(city_lookup, schema=city_lookup_schema)

3.identify corupt/invalid records

In [5]:

sales_marked = sales_raw.withColumn("amount_str_norm", F.trim(F.col("amount"))) \
    .withColumn(
        "amount_is_invalid",
        (F.col("amount_str_norm").isNull()) |
        (F.col("amount_str_norm") == "") |
        (F.lower(F.col("amount_str_norm")) == "invalid") |
        (~F.col("amount_str_norm").rlike("^[0-9]+$"))
    )

invalid_sales_records = sales_marked.filter(F.col("amount_is_invalid"))


Part 2
Cleaning and transformation

In [7]:

clean1 = sales_marked.select(
    F.trim(F.col("txn_id")).alias("txn_id"),
    F.upper(F.trim(F.col("city"))).alias("city"),          # UPPER for joins/partitioning
    F.initcap(F.trim(F.col("product"))).alias("product"),
    F.trim(F.col("category")).alias("category"),
    F.col("amount_str_norm").alias("amount_str"),
    F.trim(F.col("txn_date")).alias("txn_date_str"),
    F.trim(F.col("status")).alias("status_raw")
)


catagory to uppercase

In [8]:
clean2 = clean1.withColumn("category", F.upper(F.col("category")))

Amount to integer

In [9]:

clean3 = clean2.withColumn(
    "amount",
    F.when(F.col("amount_str").rlike("^[0-9]+$"), F.col("amount_str").cast(IntegerType()))
     .otherwise(F.lit(None).cast(IntegerType()))
).drop("amount_str")


Handle invalid/null amounts

In [10]:

amount_null_df = clean3.filter(F.col("amount").isNull())
clean4 = clean3.filter(F.col("amount").isNotNull())


parse multiple data formats

In [26]:
parsed_date = F.coalesce(
    F.try_to_timestamp(F.col("txn_date_str"), F.lit("yyyy-MM-dd")).cast(DateType()),
    F.try_to_timestamp(F.col("txn_date_str"), F.lit("dd/MM/yyyy")).cast(DateType()),
    F.try_to_timestamp(F.col("txn_date_str"), F.lit("yyyy/MM/dd")).cast(DateType()),
    F.try_to_timestamp(F.col("txn_date_str"), F.lit("dd-MM-yyyy")).cast(DateType())
)
clean5 = clean4.withColumn("txn_date", parsed_date).drop("txn_date_str")
date_null_df = clean5.filter(F.col("txn_date").isNull())
clean6 = clean5.filter(F.col("txn_date").isNotNull())

Remove Duplicates

In [12]:
clean7 = clean6.dropDuplicates(["txn_id"])

Keep only Completed transactions

In [13]:

cleaned_sales = clean7.withColumn("status_norm", F.upper(F.col("status_raw"))) \
    .filter(F.col("status_norm").startswith("COMP")) \
    .drop("status_raw", "status_norm")


Part 3
Enrichment and joins

In [14]:

city_lookup_df = city_lookup_raw.select(
    F.upper(F.col("city")).alias("city"),
    F.col("tier")
)

enriched_sales = cleaned_sales.join(
    F.broadcast(city_lookup_df), on="city", how="left"
)


In [15]:
enriched_sales.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [city])
:- Project [txn_id#14, city#15, product#16, category#21, amount#22, txn_date#23]
:  +- Filter StartsWith(status_norm#24, COMP)
:     +- Project [txn_id#14, city#15, product#16, category#21, status_raw#20, amount#22, txn_date#23, upper(status_raw#20) AS status_norm#24]
:        +- Deduplicate [txn_id#14]
:           +- Filter isnotnull(txn_date#23)
:              +- Project [txn_id#14, city#15, product#16, category#21, status_raw#20, amount#22, txn_date#23]
:                 +- Project [txn_id#14, city#15, product#16, category#21, txn_date_str#19, status_raw#20, amount#22, coalesce(to_date(txn_date_str#19, Some(yyyy-MM-dd), Some(Etc/UTC), true), to_date(txn_date_str#19, Some(dd/MM/yyyy), Some(Etc/UTC), true), to_date(txn_date_str#19, Some(yyyy/MM/dd), Some(Etc/UTC), true), to_date(txn_date_str#19, Some(dd-MM-yyyy), Some(Etc/UTC), true)) AS txn_date#23]
:                    +- Filter isnotnull(amount#22)
:                      

Part 4
revenue per city

In [16]:
rev_by_city = enriched_sales.groupBy("city").agg(F.sum("amount").alias("total_revenue"))

Revenue per product

In [17]:

rev_by_product = enriched_sales.groupBy("product").agg(F.sum("amount").alias("total_revenue"))


Rank cities by total revenue

In [18]:

city_rank = rev_by_city.select(
    "city", "total_revenue",
    F.dense_rank().over(Window.orderBy(F.desc("total_revenue"))).alias("city_rank")
)


Rank products within each city

In [19]:

rev_city_product = enriched_sales.groupBy("city", "product").agg(
    F.sum("amount").alias("product_revenue")
)
product_rank_in_city = rev_city_product.select(
    "city", "product", "product_revenue",
    F.dense_rank().over(
        Window.partitionBy("city").orderBy(F.desc("product_revenue"))
    ).alias("product_rank_in_city")
)


 Top-performing city per day

In [20]:

daily_city_rev = enriched_sales.groupBy("txn_date", "city").agg(F.sum("amount").alias("daily_revenue"))
top_city_per_day = daily_city_rev.select(
    "txn_date", "city", "daily_revenue",
    F.dense_rank().over(
        Window.partitionBy("txn_date").orderBy(F.desc("daily_revenue"))
    ).alias("rank_in_day")
).filter(F.col("rank_in_day") == 1)


Part 5

In [22]:

spark.catalog.clearCache()
enriched_sales.cache()
enriched_sales.count()


DateTimeException: [CANNOT_PARSE_TIMESTAMP] Text '09-01-2024' could not be parsed at index 0. Use `try_to_timestamp` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007

In [23]:

sales_by_city_partitioned = enriched_sales.repartition("city")
sales_by_city_partitioned.explain(True)


== Parsed Logical Plan ==
'RepartitionByExpression ['city]
+- Project [city#15, txn_id#14, product#16, category#21, amount#22, txn_date#23, tier#11]
   +- Join LeftOuter, (city#15 = city#25)
      :- Project [txn_id#14, city#15, product#16, category#21, amount#22, txn_date#23]
      :  +- Filter StartsWith(status_norm#24, COMP)
      :     +- Project [txn_id#14, city#15, product#16, category#21, status_raw#20, amount#22, txn_date#23, upper(status_raw#20) AS status_norm#24]
      :        +- Deduplicate [txn_id#14]
      :           +- Filter isnotnull(txn_date#23)
      :              +- Project [txn_id#14, city#15, product#16, category#21, status_raw#20, amount#22, txn_date#23]
      :                 +- Project [txn_id#14, city#15, product#16, category#21, txn_date_str#19, status_raw#20, amount#22, coalesce(to_date(txn_date_str#19, Some(yyyy-MM-dd), Some(Etc/UTC), true), to_date(txn_date_str#19, Some(dd/MM/yyyy), Some(Etc/UTC), true), to_date(txn_date_str#19, Some(yyyy/MM/dd), Some(E

Part 6

In [27]:
enriched_sales.write.mode("overwrite").parquet("/tmp/curated/sales_clean.parquet")

DateTimeException: [CANNOT_PARSE_TIMESTAMP] Text '09-01-2024' could not be parsed at index 0. Use `try_to_timestamp` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007

In [29]:

from pyspark.sql.utils import AnalysisException
try:
    # Demonstration: incorrect type for txn_id
    bad_schema = StructType([StructField("txn_id", IntegerType(), True)])
    spark.createDataFrame(sales_data, schema=bad_schema).show()
except AnalysisException as e:
    print("AnalysisException caught:", e)


PySparkValueError: [FIELD_STRUCT_LENGTH_MISMATCH] Length of object (7) does not match with length of fields (1).

In [30]:

enriched_sales.explain()
rev_by_city.explain()
sales_by_city_partitioned.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [city#27, txn_id#14, product#29, category#31, amount#35, txn_date#37, tier#11]
   +- BroadcastHashJoin [city#27], [city#25], LeftOuter, BuildRight, false
      :- Project [txn_id#14, city#27, product#29, category#31, amount#35, txn_date#37]
      :  +- Filter (isnotnull(status_raw#33) AND StartsWith(upper(status_raw#33), COMP))
      :     +- SortAggregate(key=[txn_id#14], functions=[first(city#15, false), first(product#16, false), first(category#21, false), first(status_raw#20, false), first(amount#22, false), first(txn_date#23, false)])
      :        +- Sort [txn_id#14 ASC NULLS FIRST], false, 0
      :           +- Exchange hashpartitioning(txn_id#14, 8), ENSURE_REQUIREMENTS, [plan_id=54]
      :              +- SortAggregate(key=[txn_id#14], functions=[partial_first(city#15, false), partial_first(product#16, false), partial_first(category#21, false), partial_first(status_raw#20, false), partial_first(amount#22, fal

In [ ]:

raw_count = sales_raw.count()
cleaned_count = enriched_sales.count()
dupe_dropped = clean6.count() - clean7.count()
invalid_amount_count = amount_null_df.count()
invalid_date_count = date_null_df.count()

print("Raw sales rows:", raw_count)
print("Cleaned sales rows:", cleaned_count)
print("Duplicates dropped:", dupe_dropped)
print("Invalid amount rows quarantined:", invalid_amount_count)
print("Invalid date rows quarantined:", invalid_date_count)
